# Free Colab cloud pipeline

All models run on the cloud GPU. No payment is required. See docs/cloud.md in the repository.

Select Runtime > Change runtime type > T4 GPU. Free sessions can disconnect and are not guaranteed to finish unattended; do not use keep-alive tricks. Optional Drive persistence is configured below.


In [ ]:
import subprocess
subprocess.run(["nvidia-smi"], check=True)


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

repo_dir = Path("/content/optimal-stopping-in-autoregressive-thought-chain")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "https://github.com/mengtobby/optimal-stopping-in-autoregressive-thought-chain.git", str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[model]"], check=True)


In [ ]:
USE_DRIVE = False  # Set True before running to retain shards across disconnects.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
workdir = Path("/content/drive/MyDrive/osc-runs/math500") if USE_DRIVE else Path("/content/osc-runs/math500")


In [ ]:
command = [sys.executable, "scripts/run_pipeline.py",
    "--workdir", str(workdir),
    "--model", "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "--dataset", "math500", "--limit", "30", "--max-hours", "3",
    "--dtype", "float16", "--n-forced", "2", "--k-siblings", "2",
    "--sibling-stride", "4", "--sibling-max-tokens", "128",
    "--max-thinking-tokens", "2048", "--max-steps", "16"]
# GPU 0 holds the small model; this avoids spending memory on automatic sharding.
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
completed = subprocess.run(command, env=env)
print("Pipeline exit status:", completed.returncode)


In [ ]:
from IPython.display import Markdown, display
display(Markdown((workdir / "SUMMARY.md").read_text()))
print("Download:", workdir / "results.zip")
print("Retain the entire workdir, including traces/shards, for collection resume.")
